# Stage 2 Notebook 62 - Exp2GGG Anchor + cls_separate_path + VFL + full 70K + 12 epochs

**Combine NB58's small cls win with NB60's geometry champion setup.** NB58 showed `cls_separate_path=True` gives gap=0.011 (vs 0.002 baseline) and val_lane_f1=0.099 (vs 0.000). Tiny improvement but real, on 3K data.

Exp2GGG: keep `cls_separate_path=True` and scale up to NB60's recipe (full 70K, bb_throttle=0.01, 12 epochs, dynamic_k matching). Tests whether the disjoint cls feature pathway scales: maybe the small win on 3K becomes a meaningful win on 70K.

Single diff vs NB60 (exp55): `cls_separate_path: false -> true`. Adds ~5% lane head params but no other change.

### Run mode
1. Smoke.
2. 12 epochs full 70K. ~2.5-3 hr wall-clock.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint_smoke.log
OK exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.6073 det_loss=3.4954 grad_cos=0.0468 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4973030388355255, 'gate/lane_mean': 0.49811992049217224, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12'
    EPOCHS = 12
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint_full12 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint_full12.tar --epochs 12 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint_full12.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp57_rmt_gca_anchor_cls_sep_vfl_full_data_long12_joint_full12_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp57_rmt

0

## What to watch in Exp2GGG

Reference NB60 (full + bb_throttle + 12 ep): matched_iou=0.554, val_lane_best_f1=0.119, gap=0.021, decoded_f1=0.050.
Reference NB58 (3K + cls_sep + topk=3): matched_iou=0.374, val_lane_best_f1=0.147, gap=0.011, decoded_f1=0.047.

Pass criteria at epoch 12:
- val/matched_line_iou >= 0.55 (match NB60).
- **val/lane_best_f1 >= 0.20** -- cls_separate_path at full data scale gives 2x the best_f1 of NB60.
- pos_score - neg_score >= 0.05 (5x NB60's 0.021).
- val/lane/decoded_f1 >= 0.10.